# Stack Overflow Error Assistant — Demo

**Author:** Juan Jesús Gómez Ortiz · MSc in Big Data Science, University of Navarra

`get_SOS_help(command)` runs a Python command and:

- prints **"works as intended"** if there is no error;
- otherwise prints the error, the **first related Stack Overflow question** and its **top-voted answer**, and opens the question in the browser.

The data comes from the official **Stack Exchange API** (no scraping, no key needed). The code lives in [`sos_help/core.py`](../sos_help/core.py).

> In this demo `open_browser=False` so the notebook can run end to end without opening tabs. In normal use, leave the default (`True`).

In [1]:
import sys
sys.path.insert(0, "..")          # make the package importable from notebooks/

import numpy as np
from sos_help import get_SOS_help

## 1. A command that works

In [2]:
get_SOS_help("np.random.uniform(-1, 1, size=100)", namespace=globals(), open_browser=False);

works as intended


## 2. Wrong keyword argument (`siz` instead of `size`)

Expected link: https://stackoverflow.com/questions/72537485/typeerror-uniform-got-an-unexpected-keyword-argument-low-size

In [3]:
get_SOS_help("np.random.uniform(-1, 1, siz=100)", namespace=globals(), open_browser=False);

TypeError: uniform() got an unexpected keyword argument 'siz'


https://stackoverflow.com/questions/72537485/typeerror-uniform-got-an-unexpected-keyword-argument-low-size
[1 votes]
We must use random.uniform while importing the related class with an alias (using 'as') or else just use import numpy while importing
An example for using alias is :
from numpy import random as np_random

Then utilize np_random.uniform()
(figured it out and had to use this to solve the problem - note I ran it on colab)


## 3. Undefined name (`nn` instead of `np`)

In [4]:
get_SOS_help("nn.random.uniform(-1, 1, size=100)", namespace=globals(), open_browser=False);

NameError: name 'nn' is not defined


https://stackoverflow.com/questions/56633138/nameerror-name-nn-is-not-defined


[21 votes]
If you got this error you can fix it with the following code:
import torch
import torch.nn as nn

You need to include both lines, since if you set just the second one it may not work if the torch package is not imported.
Where torch and torch.nn (or just nn) are two of the main PyTorch packages. You can help(torch.nn) to confirm this.
It is not uncommon when you include nn to include the functional interface as F like this:
import torch
import torch.nn as nn
import torch.nn.functional as F

To bring you the hints what you imported or what is inside the nn package I provided the list:
['AdaptiveAvgPool1d', 'AdaptiveAvgPool2d', 'AdaptiveAvgPool3d', 'AdaptiveLogSoftmaxWithLoss', 'AdaptiveMaxPool1d', 'AdaptiveMaxPool2d', 'AdaptiveMaxPool3d', 'AlphaDropout', 'AvgPool1d', 'AvgPool2d', [...]


## How it works

1. `exec` runs the command inside `try/except`.
2. The error is formatted as `ErrorType: message` and sent to the `/search/advanced` endpoint of the Stack Exchange API (sorted by relevance, only questions with answers).
3. If nothing is found, the search is repeated without the quoted names (`'siz'`, `'nn'`), which are usually the user's own typos and make the query too specific.
4. The answers of the first question are requested sorted by votes; the HTML body is cleaned with BeautifulSoup.
5. `webbrowser` opens the question with the answers sorted by score.

## Difficulties found

- The first version opened the question with **Selenium** to read the answers from the page, but Cloudflare's "Verify you are human" check blocked the automated browser again and again. Switching to the **official API** solved it: it is faster, more stable and allowed. The link is still opened in the user's normal browser, where there is no captcha.
- With the full error message the search sometimes returned nothing because of names from my own code (`'siz'`, `'nn'`), hence the fallback search without them.